# Lab 03-07: Multi-Agent Systems and Genie

| | |
|---|---|
| **Module** | 03 -- Application Development (30% of exam) |
| **Estimated Questions** | ~13 questions |
| **UI Sections** | Genie \| AI/BI Dashboards \| Playground |
| **Estimated Time** | 40--50 minutes |
| **Difficulty** | Intermediate |

---

## What Are Multi-Agent Systems and Why Does Genie Matter?

A **multi-agent system** divides complex tasks among specialized agents, each
with its own tools, instructions, and expertise. Instead of one monolithic
chain that tries to do everything, you build focused specialists and an
orchestrator that routes queries to the right one.

**Genie** is Databricks' natural-language-to-SQL interface. Users type plain
English questions, and Genie generates SQL queries against your Delta tables.
It is a specialized data agent that fits naturally into multi-agent architectures.

The exam tests:
- Understanding multi-agent architecture patterns (sequential, parallel, hierarchical)
- Knowing what Genie does and how to set up a Genie Space
- Knowing when to route to Genie vs custom agents
- Understanding how MCP (Model Context Protocol) servers enable tool integration

---

## Mental Model

> **Analogy:** A multi-agent system is like a hospital ER. The triage nurse
> (orchestrator) does not treat anyone -- they route patients to the right
> specialist (agent). The cardiologist, surgeon, and radiologist each have
> their own expertise and tools. Genie is the data specialist -- it translates
> plain English into SQL.
>
> When a patient (query) needs structured data analysis ("What were sales last
> quarter?"), the triage nurse routes them to Genie, who translates the plain
> English question into a SQL query and returns the results.
>
> The cardiologist (RAG agent) handles knowledge-base questions. The surgeon
> (code agent) handles code generation. Each agent has its own specialized
> tools and instructions. The orchestrator's only job is routing.

---

## Exam Alert -- Common Traps

| Trap | Why Students Fall For It | The Truth |
|---|---|---|
| "Genie writes Python code" | Genie feels like a general AI assistant | Genie generates **SQL queries** against your Delta tables. It does not write Python, generate reports, or create visualizations directly. |
| "Multi-agent systems require a specific framework" | Students look for one right answer | You can build multi-agent systems with **LangChain, custom code, or Agent Bricks (Multiagent Supervisor)**. There is no single required framework. |
| "All agents share the same tools" | It seems simpler | Each agent has its own **specialized tools and instructions**. A SQL agent has SQL tools; a search agent has retrieval tools. Sharing everything defeats the purpose of specialization. |
| "Genie can query any data source" | Genie seems powerful | Genie queries **Delta tables that you explicitly configure** in a Genie Space. It does not access external databases, APIs, or unstructured data. |
| "The orchestrator is the smartest agent" | The orchestrator makes routing decisions | The orchestrator is often the **simplest** component -- it just classifies the query and routes it. The specialist agents do the hard work. |

---

## Prerequisites

- Completed Lab 03-06 (Agent Framework with MLflow)
- Understanding of LangChain chain composition
- Familiarity with SQL basics

---

## Exam Objectives Covered

- Understand multi-agent system architecture patterns
- Know what Genie does and how to configure Genie Spaces
- Route queries between different specialized agents
- Understand MCP servers for tool integration
- Know when to use Genie vs RAG vs custom agents

---

## Setup

In [ ]:
# ------------------------------------------------------------------
# Setup: Install dependencies
# ------------------------------------------------------------------

%pip install openai --quiet

# Restart Python to pick up the new package
dbutils.library.restartPython()


**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

In [ ]:
# ------------------------------------------------------------------
# Setup: Initialize the OpenAI-compatible client for Databricks
# ------------------------------------------------------------------

from openai import OpenAI
import os

client = OpenAI(
    api_key=os.environ.get("DATABRICKS_TOKEN"),
    base_url=f"{os.environ.get('DATABRICKS_HOST')}/serving-endpoints"
)

MODEL = "databricks-meta-llama-3-3-70b-instruct"
CATALOG = "main"
SCHEMA = "genai_labs"


def chat(user_msg, system_msg="You are a helpful assistant.", temperature=0.0, max_tokens=512):
    """Send a chat completion request and return the response text."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content


print("Client ready.  Model:", MODEL)


**Expected output:**
```
Client ready.  Model: databricks-meta-llama-3-3-70b-instruct
```

---

## Step 1: Multi-Agent Architecture Patterns

Multi-agent systems come in three primary patterns. The exam tests whether
you can identify which pattern fits a given scenario.

### Pattern 1: Sequential (Pipeline)

```
  Query --> Agent A --> Agent B --> Agent C --> Response
```

Each agent processes the output of the previous agent. Like an assembly line.

**Use when:** Each step depends on the previous step's output.

**Example:** Extract entities (Agent A) --> Validate entities (Agent B) --> Generate report (Agent C)

### Pattern 2: Parallel (Fan-out / Fan-in)

```
                +--> Agent A --+
                |              |
  Query --> Router            Aggregator --> Response
                |              |
                +--> Agent B --+
```

Multiple agents process the query simultaneously. Results are aggregated.

**Use when:** The query needs multiple independent perspectives.

**Example:** Search internal docs (Agent A) AND search external knowledge (Agent B) --> Combine results

### Pattern 3: Hierarchical (Orchestrator + Specialists)

```
                  +-- SQL Agent (Genie)
                  |
  Query --> Orchestrator -- RAG Agent
                  |
                  +-- Code Agent
```

An orchestrator classifies the query and routes it to the right specialist.
This is the **most common pattern on the exam**.

**Use when:** Different query types need different capabilities.

**Example:** Data questions --> Genie, knowledge questions --> RAG, code questions --> Code agent

In [ ]:
# ------------------------------------------------------------------
# Step 1: Visualize the three multi-agent patterns
# ------------------------------------------------------------------

print("=" * 76)
print("MULTI-AGENT ARCHITECTURE PATTERNS")
print("=" * 76)

print()
print("PATTERN 1: SEQUENTIAL (Pipeline)")
print("=" * 40)
print()
print("  User Query")
print("      |")
print("      v")
print("  +----------+     +----------+     +----------+")
print("  | Agent A  | --> | Agent B  | --> | Agent C  |")
print("  | Extract  |     | Validate |     | Generate |")
print("  +----------+     +----------+     +----------+")
print("                                         |")
print("                                         v")
print("                                     Response")
print()
print("  - Each agent depends on the previous agent's output")
print("  - Like an assembly line: each step adds value")
print("  - Failure in one step stops the pipeline")

print()
print()
print("PATTERN 2: PARALLEL (Fan-out / Fan-in)")
print("=" * 40)
print()
print("                  +------------+")
print("              +-> | Agent A    | -+")
print("              |   | (internal) |  |")
print("  User Query  |   +------------+  |   +------------+")
print("      |       |                   +-> | Aggregator | -> Response")
print("      +-------+                   +-> |            |")
print("              |   +------------+  |   +------------+")
print("              +-> | Agent B    | -+")
print("                  | (external) |")
print("                  +------------+")
print()
print("  - Agents run simultaneously on the same query")
print("  - Results are combined by an aggregator")
print("  - Faster than sequential for independent tasks")

print()
print()
print("PATTERN 3: HIERARCHICAL (Orchestrator + Specialists)  ** MOST COMMON **")
print("=" * 60)
print()
print("                       +------------------+")
print("                   +-> | SQL Agent (Genie)|")
print("                   |   +------------------+")
print("  User Query       |")
print("      |            |   +------------------+")
print("      v            +-> | RAG Agent        |")
print("  +-------------+  |   +------------------+")
print("  | Orchestrator |--+")
print("  | (Router)     |  |   +------------------+")
print("  +-------------+  +-> | Code Agent       |")
print("                       +------------------+")
print()
print("  - Orchestrator classifies the query type")
print("  - Routes to the specialist with the right tools")
print("  - Each specialist has different tools and prompts")
print("  - THIS IS THE PATTERN TESTED MOST ON THE EXAM")


**Expected output:** The three architecture diagrams printed above, showing
sequential, parallel, and hierarchical patterns with explanations.

**Key exam point:** The hierarchical pattern with an orchestrator routing to
specialists is the most commonly tested pattern. When the exam describes a
system with "different types of queries going to different agents," this is
the pattern they are describing.

---

## Step 2: Building a Simple Orchestrator (Router)

The orchestrator is the "triage nurse" -- it examines the incoming query and
decides which specialist agent should handle it. In practice, this is often
just a classification step using an LLM.

### How the Orchestrator Works

```
1. Receive the user query
2. Classify the query type (SQL, knowledge, code, general)
3. Route to the appropriate specialist agent
4. Return the specialist's response
```

The orchestrator does NOT answer the question itself. Its only job is routing.

In [ ]:
# ------------------------------------------------------------------
# Step 2: Build an orchestrator that classifies and routes queries
# ------------------------------------------------------------------

import json


def classify_query(query: str) -> str:
    """Use the LLM to classify a query into one of the agent categories."""

    classification_prompt = f"""Classify the following user query into exactly one category.
Return ONLY the category name, nothing else.

Categories:
- SQL_DATA: Questions about structured data, metrics, numbers, sales, counts, aggregations
- KNOWLEDGE: Questions about policies, procedures, documentation, how things work
- CODE: Requests to write, debug, or explain code
- GENERAL: Everything else (greetings, clarifications, off-topic)

Query: "{query}"
Category:"""

    result = chat(classification_prompt, temperature=0.0, max_tokens=20)
    return result.strip()


def orchestrate(query: str) -> dict:
    """Route a query to the appropriate specialist agent."""

    # Step 1: Classify
    category = classify_query(query)

    # Step 2: Route to specialist
    routes = {
        "SQL_DATA": "sql_agent",
        "KNOWLEDGE": "rag_agent",
        "CODE": "code_agent",
        "GENERAL": "general_agent",
    }

    agent = routes.get(category, "general_agent")

    return {
        "query": query,
        "category": category,
        "routed_to": agent,
    }


# Test the orchestrator with different query types
TEST_QUERIES = [
    "What were total sales last quarter?",
    "What is our company's refund policy?",
    "Write a Python function to calculate Fibonacci numbers",
    "Hello, how are you today?",
    "How many customers signed up in January?",
    "Explain how RAG works in Databricks",
]

print("=" * 76)
print("ORCHESTRATOR ROUTING TEST")
print("=" * 76)

for query in TEST_QUERIES:
    result = orchestrate(query)
    print(f"\n  Query:    {result['query']}")
    print(f"  Category: {result['category']}")
    print(f"  Route:    {result['routed_to']}")

print(f"\n{'=' * 76}")
print("The orchestrator classified each query and routed it to the")
print("appropriate specialist. It did NOT answer any questions itself.")
print("=" * 76)


**Expected output:**
```
============================================================================
ORCHESTRATOR ROUTING TEST
============================================================================

  Query:    What were total sales last quarter?
  Category: SQL_DATA
  Route:    sql_agent

  Query:    What is our company's refund policy?
  Category: KNOWLEDGE
  Route:    rag_agent

  Query:    Write a Python function to calculate Fibonacci numbers
  Category: CODE
  Route:    code_agent

  Query:    Hello, how are you today?
  Category: GENERAL
  Route:    general_agent

  Query:    How many customers signed up in January?
  Category: SQL_DATA
  Route:    sql_agent

  Query:    Explain how RAG works in Databricks
  Category: KNOWLEDGE
  Route:    rag_agent

============================================================================
The orchestrator classified each query and routed it to the
appropriate specialist. It did NOT answer any questions itself.
============================================================================
```

**What just happened:** We built an orchestrator using an LLM as the classifier.
The LLM reads the query and decides which category it belongs to. The orchestrator
then routes to the matching specialist. This is the hierarchical pattern in action.

---

## Step 3: Implementing Specialist Agents

Each specialist agent has its own:
- **System prompt** (role and instructions)
- **Tools** (functions it can call)
- **Domain knowledge** (what it is expert in)

Let's build three specialist agents to complete our multi-agent system.

In [ ]:
# ------------------------------------------------------------------
# Step 3: Implement specialist agents with their own prompts and tools
# ------------------------------------------------------------------

def sql_agent(query: str) -> str:
    """Specialist agent for data/SQL questions.
    In production, this would be Genie or a SQL-generating agent.
    """
    system = (
        "You are a SQL data analyst agent. When users ask about data, "
        "metrics, or numbers, generate a SQL query that would answer "
        "their question. Assume the database has these tables:\n"
        "- main.genai_labs.sales (date, product, region, amount, quantity)\n"
        "- main.genai_labs.customers (customer_id, name, signup_date, region)\n"
        "- main.genai_labs.orders (order_id, customer_id, order_date, total)\n\n"
        "Return the SQL query and a brief explanation."
    )
    return chat(query, system_msg=system, max_tokens=300)


def rag_agent(query: str) -> str:
    """Specialist agent for knowledge/documentation questions.
    In production, this would use vector search for retrieval.
    """
    system = (
        "You are a knowledge base agent for a technology company. "
        "Answer questions about policies, procedures, and how things work. "
        "If you do not know the answer, say so clearly. "
        "Keep answers concise -- 2-3 sentences maximum."
    )
    return chat(query, system_msg=system, max_tokens=200)


def code_agent(query: str) -> str:
    """Specialist agent for code-related requests."""
    system = (
        "You are a Python coding assistant. Write clean, well-commented "
        "code. Include type hints and docstrings. Keep solutions concise."
    )
    return chat(query, system_msg=system, max_tokens=400)


def general_agent(query: str) -> str:
    """Fallback agent for general/off-topic queries."""
    system = (
        "You are a helpful assistant. For greetings, respond warmly. "
        "For off-topic questions, politely redirect to available services: "
        "data analysis, knowledge lookup, or code assistance."
    )
    return chat(query, system_msg=system, max_tokens=150)


# Map agent names to functions
AGENTS = {
    "sql_agent": sql_agent,
    "rag_agent": rag_agent,
    "code_agent": code_agent,
    "general_agent": general_agent,
}


def run_multi_agent(query: str) -> dict:
    """Complete multi-agent pipeline: classify, route, execute."""
    # Step 1: Orchestrate (classify + route)
    routing = orchestrate(query)

    # Step 2: Execute the specialist agent
    agent_fn = AGENTS[routing["routed_to"]]
    response = agent_fn(query)

    return {
        **routing,
        "response": response,
    }


# Test the full pipeline
print("=" * 76)
print("MULTI-AGENT SYSTEM -- FULL PIPELINE TEST")
print("=" * 76)

test_cases = [
    "What were total sales by region last month?",
    "What is our return policy for electronics?",
    "Write a function to validate email addresses in Python",
]

for query in test_cases:
    result = run_multi_agent(query)
    print(f"\n{'~' * 76}")
    print(f"  Query:    {result['query']}")
    print(f"  Category: {result['category']}")
    print(f"  Agent:    {result['routed_to']}")
    print(f"  Response:")
    for line in result['response'].split('\n'):
        print(f"    {line}")


**Expected output:**
```
============================================================================
MULTI-AGENT SYSTEM -- FULL PIPELINE TEST
============================================================================

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Query:    What were total sales by region last month?
  Category: SQL_DATA
  Agent:    sql_agent
  Response:
    SELECT region, SUM(amount) as total_sales
    FROM main.genai_labs.sales
    WHERE date >= DATE_TRUNC('month', CURRENT_DATE - INTERVAL 1 MONTH)
    ...

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Query:    What is our return policy for electronics?
  Category: KNOWLEDGE
  Agent:    rag_agent
  Response:
    Our return policy allows returns within 30 days of purchase...

~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
  Query:    Write a function to validate email addresses in Python
  Category: CODE
  Agent:    code_agent
  Response:
    import re
    def validate_email(email: str) -> bool:
        ...
```

**What just happened:** We built a complete multi-agent system:
1. The **orchestrator** classified each query
2. The query was **routed** to the right specialist
3. The specialist **generated a response** using its own system prompt and expertise

Each specialist produced domain-appropriate output -- SQL for data questions,
concise answers for knowledge questions, and code for programming requests.

---

## Step 4: Genie -- Natural Language to SQL

**Genie** is Databricks' built-in natural-language-to-SQL interface. It lets
non-technical users ask questions in plain English and get answers from
structured Delta tables.

### What Genie Does

```
User: "What were total sales last quarter?"
                    |
                    v
              +----------+
              |  Genie   |  (understands table schemas, column meanings)
              +----------+
                    |
                    v
  SELECT SUM(amount) FROM sales
  WHERE date BETWEEN '2024-10-01' AND '2024-12-31'
                    |
                    v
  Result: $4,235,891.00
```

### What Genie Does NOT Do

- Does NOT write Python code
- Does NOT query unstructured data or vector search indexes
- Does NOT create visualizations (use AI/BI Dashboards for that)
- Does NOT access external databases or APIs
- Does NOT work without a configured **Genie Space**

### Setting Up a Genie Space

A **Genie Space** is a configured environment that tells Genie which tables
it can query and provides context about the data.

**UI \u2192** Left nav \u2192 **Genie** \u2192 **New Genie Space**

1. **Name** your Genie Space (e.g., "Sales Analytics")
2. **Add tables**: Select the Delta tables Genie can query
   (e.g., `main.genai_labs.sales`, `main.genai_labs.customers`)
3. **Add instructions**: Describe what the data means
   (e.g., "The 'amount' column in the sales table represents revenue in USD")
4. **Add sample questions**: Provide example queries so Genie learns the patterns
   (e.g., "What were total sales last month?" --> expected SQL)
5. **Configure permissions**: Control who can access this Genie Space
6. Click **Save**

### Why Instructions and Sample Questions Matter

Genie generates SQL by understanding:
- The **table schemas** (column names and types)
- Your **instructions** (business context -- "fiscal year starts in April")
- Your **sample questions** (patterns -- "when users say 'revenue', they mean the 'amount' column")

Without good instructions, Genie may generate technically valid SQL that
produces wrong business results (e.g., using calendar year instead of fiscal year).

> **Exam tip:** If a question asks "how do you improve Genie's accuracy?"
> the answer is: add better instructions and more sample questions to the
> Genie Space. NOT: fine-tune the model or change the temperature.

In [ ]:
# ------------------------------------------------------------------
# Step 4: Simulate Genie's natural-language-to-SQL behavior
# ------------------------------------------------------------------
# Genie is a UI-based tool, but we can demonstrate its core logic
# by using an LLM with a SQL-focused system prompt and table context.

GENIE_SYSTEM_PROMPT = (
    "You are Genie, a natural-language-to-SQL assistant for Databricks.\n\n"
    "AVAILABLE TABLES:\n"
    "- main.genai_labs.sales (date DATE, product STRING, region STRING, amount DOUBLE, quantity INT)\n"
    "- main.genai_labs.customers (customer_id INT, name STRING, signup_date DATE, region STRING)\n"
    "- main.genai_labs.orders (order_id INT, customer_id INT, order_date DATE, total DOUBLE)\n\n"
    "BUSINESS CONTEXT:\n"
    "- Fiscal year starts January 1\n"
    "- 'Revenue' means the 'amount' column in the sales table\n"
    "- Regions are: North, South, East, West\n"
    "- 'New customers' means customers whose signup_date is within the specified period\n\n"
    "RULES:\n"
    "- Generate ONLY SQL queries. No Python, no explanations unless asked.\n"
    "- Use Databricks SQL syntax (Delta Lake compatible).\n"
    "- Always qualify table names with catalog.schema (main.genai_labs.xxx).\n"
    "- If the question is ambiguous, state your assumption before the SQL."
)

GENIE_QUERIES = [
    "What were total sales last month?",
    "Show me the top 5 customers by order total",
    "How many new customers signed up in Q1?",
    "What is the average order value by region?",
]

print("=" * 76)
print("GENIE SIMULATION -- Natural Language to SQL")
print("=" * 76)

for query in GENIE_QUERIES:
    sql = chat(query, system_msg=GENIE_SYSTEM_PROMPT, max_tokens=200)
    print(f"\n  User: \"{query}\"")
    print(f"  Genie SQL:")
    for line in sql.strip().split("\n"):
        print(f"    {line}")
    print(f"  {'-' * 60}")


**Expected output:**
```
============================================================================
GENIE SIMULATION -- Natural Language to SQL
============================================================================

  User: "What were total sales last month?"
  Genie SQL:
    SELECT SUM(amount) AS total_sales
    FROM main.genai_labs.sales
    WHERE date >= DATE_TRUNC('MONTH', CURRENT_DATE - INTERVAL 1 MONTH)
      AND date < DATE_TRUNC('MONTH', CURRENT_DATE)
  ------------------------------------------------------------

  User: "Show me the top 5 customers by order total"
  Genie SQL:
    SELECT c.name, SUM(o.total) AS total_orders
    FROM main.genai_labs.orders o
    JOIN main.genai_labs.customers c ON o.customer_id = c.customer_id
    GROUP BY c.name
    ORDER BY total_orders DESC
    LIMIT 5
  ------------------------------------------------------------
  ...
```

**What just happened:** We simulated Genie's behavior -- taking natural language
questions and generating SQL queries. In the real Genie product:
- The table schemas are automatically discovered from Unity Catalog
- Business context comes from the Genie Space configuration
- Results are displayed as tables and charts in the Genie UI
- Users interact through a chat interface, not code

---

## Step 5: Genie as an Agent in a Multi-Agent System

Genie is a natural fit as the "data specialist" in a hierarchical multi-agent
system. When the orchestrator detects a structured data question, it routes
to Genie instead of trying to answer with RAG or general knowledge.

### When to Route to Genie vs Other Agents

| Query Type | Route To | Why |
|---|---|---|
| "What were sales last quarter?" | **Genie** (SQL Agent) | Structured data query over Delta tables |
| "What is our refund policy?" | **RAG Agent** | Unstructured knowledge in documents |
| "Write a Python function to..." | **Code Agent** | Code generation task |
| "Summarize this document" | **RAG Agent** or **LLM directly** | Text processing on unstructured data |
| "How many users signed up?" | **Genie** (SQL Agent) | Structured data query (count/aggregation) |
| "Explain how vector search works" | **RAG Agent** or **LLM directly** | Knowledge/explanation question |

### The Key Decision Rule

> **Structured data (numbers, counts, aggregations, joins) --> Genie (SQL)**
>
> **Unstructured knowledge (policies, docs, explanations) --> RAG Agent**
>
> If the answer lives in a Delta table, route to Genie. If the answer lives
> in a document or knowledge base, route to RAG.

In [ ]:
# ------------------------------------------------------------------
# Step 5: Routing decision quiz -- Genie vs RAG vs Other
# ------------------------------------------------------------------

ROUTING_QUIZ = [
    {
        "query": "What is the average deal size for enterprise customers?",
        "route": "Genie (SQL Agent)",
        "why": "This is a structured data query (average, filtering by customer type). The answer is in a Delta table.",
    },
    {
        "query": "What are the steps to request a refund?",
        "route": "RAG Agent",
        "why": "This is a knowledge question about a procedure/policy. The answer is in a document, not a table.",
    },
    {
        "query": "Show me monthly revenue trends for the past year",
        "route": "Genie (SQL Agent)",
        "why": "Revenue data is structured (amounts, dates). Genie generates the SQL; a dashboard can visualize it.",
    },
    {
        "query": "How does the RAG pipeline work in our system?",
        "route": "RAG Agent",
        "why": "This is a knowledge question about system architecture. The answer is in documentation.",
    },
    {
        "query": "What percentage of orders were returned last month?",
        "route": "Genie (SQL Agent)",
        "why": "Percentage calculation over structured order data. SQL can compute this directly.",
    },
    {
        "query": "What model should I use for sentiment analysis?",
        "route": "RAG Agent or LLM",
        "why": "This is an advisory/knowledge question. No structured data query is needed.",
    },
]

print("=" * 76)
print("ROUTING DECISION QUIZ: Genie vs RAG vs Other")
print("=" * 76)

for i, q in enumerate(ROUTING_QUIZ, 1):
    print(f"\n  Q{i}: \"{q['query']}\"")
    print(f"  --> Route to: {q['route']}")
    print(f"  Why:  {q['why']}")

print(f"\n{'=' * 76}")
print("DECISION RULE:")
print("  Structured data (numbers, counts, aggregations) --> Genie (SQL)")
print("  Unstructured knowledge (policies, docs, how-to)  --> RAG Agent")
print("  Code requests                                     --> Code Agent")
print("=" * 76)


**Expected output:**
```
============================================================================
ROUTING DECISION QUIZ: Genie vs RAG vs Other
============================================================================

  Q1: "What is the average deal size for enterprise customers?"
  --> Route to: Genie (SQL Agent)
  Why:  This is a structured data query (average, filtering by customer type)...

  Q2: "What are the steps to request a refund?"
  --> Route to: RAG Agent
  Why:  This is a knowledge question about a procedure/policy...

  ...

============================================================================
DECISION RULE:
  Structured data (numbers, counts, aggregations) --> Genie (SQL)
  Unstructured knowledge (policies, docs, how-to)  --> RAG Agent
  Code requests                                     --> Code Agent
============================================================================
```

---

## Step 6: MCP Servers -- Tool Integration for Agents

The **Model Context Protocol (MCP)** is an open standard that lets agents
connect to external tools and data sources through a uniform interface.
On Databricks, MCP servers extend what agents can do by providing
structured access to databases, APIs, file systems, and more.

### What Is an MCP Server?

An MCP server exposes **tools** (functions an agent can call) and **resources**
(data an agent can read) through a standardized protocol. Think of it as a
plugin system for AI agents.

```
Agent                MCP Server              External System
=====                ==========              ===============
"Look up order       Tool: get_order()       Order Database
 #12345"  -------->  (validates input,  ---> (SQL query)
                      formats output)   <--- (result row)
          <--------  {"order_id": 12345,
                      "status": "shipped"}
```

### Three Types of MCP Servers on Databricks

| Type | What It Is | Examples |
|---|---|---|
| **Managed MCP Servers** | Built into Databricks, pre-configured | Unity Catalog (tables, volumes, functions), Genie (NL-to-SQL) |
| **External MCP Servers** | Third-party servers you connect | Slack, GitHub, Jira |
| **Custom MCP Servers** | Servers you build for your own tools | Internal API wrapper, custom database connector |

### Managed MCP Servers: Unity Catalog and Genie

The **Unity Catalog MCP server** is the most important managed server. It gives
agents direct access to:
- **Tables**: Query Delta tables via SQL
- **Volumes**: Read files stored in Volumes
- **Functions**: Call Unity Catalog functions (UDFs)

The **Genie MCP server** gives agents the ability to use Genie for
natural-language-to-SQL translation, so an agent can route data questions
to Genie without building its own SQL generation logic.

### External MCP Servers

External MCP servers connect agents to third-party services:
- **Slack MCP server** -- send messages, read channels, search history
- **GitHub MCP server** -- read repos, create issues, manage pull requests
- **Jira MCP server** -- create tickets, update status, query backlogs

These servers follow the same MCP protocol, so agents interact with them
using the same interface regardless of the underlying service.

### Custom MCP Servers

For internal tools and proprietary APIs, you build custom MCP servers:
1. Implement the MCP protocol (define tools and their schemas)
2. Host the server (on Databricks or externally)
3. Register the server in the Databricks UI

### Setting Up MCP Servers

**UI \u2192** Left nav \u2192 **Playground** \u2192 **Tools** section \u2192
**Add MCP Server** \u2192 Select managed, external, or custom

For custom MCP servers, you define:
1. **Server URL**: Where the MCP server is hosted
2. **Authentication**: How the agent authenticates with the server
3. **Available tools**: Which tools the server exposes
4. **Tool descriptions**: Natural language descriptions so the agent knows when to use each tool

> **Exam tip:** MCP servers are the mechanism that gives agents access to
> external tools. The Unity Catalog MCP server is the most commonly
> referenced on the exam because it connects agents to your data lakehouse.

In [ ]:
# ------------------------------------------------------------------
# Step 6: Understanding MCP Server architecture
# ------------------------------------------------------------------
# MCP is primarily a protocol/configuration concept. We demonstrate
# the architecture and decision framework here.

print("=" * 76)
print("MCP SERVER TYPES AND USE CASES")
print("=" * 76)

MCP_TYPES = [
    {
        "type": "Managed MCP Servers",
        "description": "Built into Databricks, pre-configured and governed",
        "examples": [
            "Unity Catalog MCP Server -- access tables, volumes, functions",
            "Genie MCP Server -- natural-language-to-SQL for data questions",
        ],
        "setup": "Available out-of-the-box in the Playground and Agent Framework",
    },
    {
        "type": "External MCP Servers",
        "description": "Third-party servers that follow the MCP protocol",
        "examples": [
            "Slack MCP Server -- send messages, read channels, search history",
            "GitHub MCP Server -- read repos, create issues, manage PRs",
            "Jira MCP Server -- create tickets, update status, query backlogs",
        ],
        "setup": "Configure server URL and authentication in the Databricks UI",
    },
    {
        "type": "Custom MCP Servers",
        "description": "Servers you build for your own internal tools and APIs",
        "examples": [
            "Internal CRM API wrapper -- look up customer data",
            "Proprietary calculation engine -- run domain-specific models",
            "Legacy database connector -- query non-Delta data sources",
        ],
        "setup": "Build a server implementing the MCP protocol, host it, register in Databricks",
    },
]

for mcp in MCP_TYPES:
    print(f"\n  {mcp['type']}")
    print(f"  {'~' * len(mcp['type'])}")
    print(f"  {mcp['description']}")
    print(f"  Setup: {mcp['setup']}")
    print(f"  Examples:")
    for ex in mcp['examples']:
        print(f"    - {ex}")

print(f"\n{'=' * 76}")
print("HOW MCP FITS IN A MULTI-AGENT SYSTEM")
print("=" * 76)
print()
print("  User Query")
print("      |")
print("      v")
print("  Orchestrator (routes query)")
print("      |")
print("      +-- Agent A + Managed MCP Server (Unity Catalog)")
print("      |     |-- Tool: query_table(sql)")
print("      |     |-- Tool: read_volume(path)")
print("      |     +-- Tool: call_function(name, args)")
print("      |")
print("      +-- Agent B + Managed MCP Server (Genie)")
print("      |     +-- Tool: ask_genie(natural_language_question)")
print("      |")
print("      +-- Agent C + External MCP Server (GitHub)")
print("      |     |-- Tool: search_code(query)")
print("      |     +-- Tool: create_issue(title, body)")
print("      |")
print("      +-- Agent D + Custom MCP Server (Internal API)")
print("            |-- Tool: lookup_customer(id)")
print("            +-- Tool: calculate_risk_score(params)")
print()
print("Each agent gets its own set of tools via MCP servers.")
print("The MCP protocol ensures a uniform interface regardless")
print("of what the tools actually connect to.")


**Expected output:**
```
============================================================================
MCP SERVER TYPES AND USE CASES
============================================================================

  Managed MCP Servers
  ~~~~~~~~~~~~~~~~~~~
  Built into Databricks, pre-configured and governed
  Setup: Available out-of-the-box in the Playground and Agent Framework
  Examples:
    - Unity Catalog MCP Server -- access tables, volumes, functions
    - Genie MCP Server -- natural-language-to-SQL for data questions

  External MCP Servers
  ~~~~~~~~~~~~~~~~~~~~
  Third-party servers that follow the MCP protocol
  ...

============================================================================
HOW MCP FITS IN A MULTI-AGENT SYSTEM
============================================================================

  User Query --> Orchestrator --> Agent A + Managed MCP (Unity Catalog)
                              --> Agent B + Managed MCP (Genie)
                              --> Agent C + External MCP (GitHub)
                              --> Agent D + Custom MCP (Internal API)
  ...
```

---

## Step 7: Putting It All Together -- The Complete Multi-Agent Architecture

Let's integrate everything we have learned into a complete reference
architecture that maps to what the exam expects you to know.

In [ ]:
# ------------------------------------------------------------------
# Step 7: Complete multi-agent architecture reference
# ------------------------------------------------------------------

print("=" * 76)
print("COMPLETE MULTI-AGENT ARCHITECTURE ON DATABRICKS")
print("=" * 76)
print()
print("  +-------------------------------------------------------------------+")
print("  |                        USER INTERFACE                             |")
print("  |  (Databricks App, Playground, Slack Bot, Custom Web App)          |")
print("  +-------------------------------------------------------------------+")
print("                              |")
print("                              v")
print("  +-------------------------------------------------------------------+")
print("  |                      ORCHESTRATOR                                 |")
print("  |  - Classifies query type                                          |")
print("  |  - Routes to specialist agent                                     |")
print("  |  - Built with LangChain, Agent Framework, or custom Python        |")
print("  +-------------------------------------------------------------------+")
print("          |                |                |               |")
print("          v                v                v               v")
print("  +--------------+  +--------------+  +-------------+  +----------+")
print("  | GENIE        |  | RAG AGENT    |  | CODE AGENT  |  | GENERAL  |")
print("  | (SQL Agent)  |  |              |  |             |  | AGENT    |")
print("  +--------------+  +--------------+  +-------------+  +----------+")
print("  | NL --> SQL   |  | Vector Search|  | Code gen    |  | Chat     |")
print("  | Delta tables |  | Doc retrieval|  | Debugging   |  | Fallback |")
print("  +--------------+  +--------------+  +-------------+  +----------+")
print("          |                |                |")
print("          v                v                v")
print("  +--------------+  +--------------+  +-------------+")
print("  | MCP Server   |  | MCP Server   |  | MCP Server  |")
print("  | (UC + Genie) |  | (VS Index)   |  | (GitHub)    |")
print("  +--------------+  +--------------+  +-------------+")
print()
print("  ALL agents are:")
print("  - Logged with MLflow (Lab 03-06)")
print("  - Registered in Unity Catalog")
print("  - Served via Model Serving endpoints")
print("  - Monitored via inference tables")

# Summary comparison table
print()
print("=" * 76)
print("COMPONENT COMPARISON")
print("=" * 76)
print()
print(f"{'Component':<20s} {'What It Does':<30s} {'Key Technology'}")
print("-" * 76)
print(f"{'Orchestrator':<20s} {'Classifies and routes':<30s} {'LLM + classification prompt'}")
print(f"{'Genie':<20s} {'NL to SQL':<30s} {'Genie Space + Delta tables'}")
print(f"{'RAG Agent':<20s} {'Doc retrieval + answer':<30s} {'Vector Search + LLM'}")
print(f"{'Code Agent':<20s} {'Code generation':<30s} {'LLM + code prompt'}")
print(f"{'MCP Server':<20s} {'Tool access for agents':<30s} {'MCP protocol + UC/APIs'}")
print(f"{'MLflow':<20s} {'Logging + versioning':<30s} {'Experiments + UC registry'}")
print(f"{'Model Serving':<20s} {'Deploy as endpoints':<30s} {'REST API + auto-scaling'}")


**Expected output:** The complete architecture diagram and component comparison
table printed above, showing how all the pieces fit together.

**Key exam insight:** The exam does not expect you to implement a full
multi-agent system from scratch. It expects you to understand:
1. The **hierarchical pattern** (orchestrator + specialists)
2. **When to route to Genie** (structured data) vs RAG (unstructured knowledge)
3. **How MCP servers** provide tool access to agents
4. **How MLflow** logs and versions everything
5. That **each agent has its own tools and instructions**

---

## Exam Tips

| # | Tip | Why It Matters |
|---|---|---|
| 1 | **Genie generates SQL, not Python.** It translates natural language to SQL queries against Delta tables. | The number-one Genie trap. If the exam asks "what does Genie produce?" the answer is SQL. |
| 2 | **Genie requires a configured Genie Space.** You must specify which tables it can query and provide business context instructions. | Without a Genie Space, Genie has no tables to query. Instructions improve accuracy. |
| 3 | **The orchestrator routes, not answers.** It classifies the query type and sends it to the right specialist. | The exam may describe an orchestrator as the component that "decides which agent handles the query." |
| 4 | **Each agent has its own tools and instructions.** Agents are specialists, not generalists. | Common trap: "all agents share the same tools" is WRONG. Each agent has specialized capabilities. |
| 5 | **MCP servers give agents access to external tools.** Managed (Unity Catalog, Genie), External (Slack, GitHub), and Custom (your APIs). | The exam tests knowing that MCP is how agents connect to tools, and that the Unity Catalog MCP server is the key managed server. |

---

## Key Takeaways

### Core Concepts -- Multi-Agent Architecture
- **Sequential**: Agents process in order, each feeding the next (pipeline).
- **Parallel**: Agents process simultaneously, results aggregated (fan-out/fan-in).
- **Hierarchical**: Orchestrator routes to specialists (most common on the exam).
- The orchestrator is a **classifier/router**, not an answerer.

### Databricks Specifics -- Genie and MCP
- Genie translates **natural language to SQL** against Delta tables.
- Genie requires a **Genie Space** with configured tables, instructions, and sample questions.
- Genie does NOT write Python, query unstructured data, or create visualizations.
- Improve Genie accuracy by adding better instructions and sample questions.
- Route to Genie for **structured data** (numbers, counts, aggregations).
- **Managed MCP servers**: Unity Catalog (tables, volumes, functions) and Genie (NL-to-SQL).
- **External MCP servers**: Slack, GitHub, Jira -- connect via URL and authentication.
- **Custom MCP servers**: Your own servers for internal APIs and tools.
- MCP provides a **uniform protocol** for agent-to-tool communication.

### Exam Essentials -- Routing and Integration
- **Structured data** (numbers, metrics, counts) --> Genie / SQL Agent.
- **Unstructured knowledge** (policies, docs, explanations) --> RAG Agent.
- **Code requests** --> Code Agent.
- Multi-agent systems can be built with **LangChain, custom Python, or Agent Bricks (Multiagent Supervisor)** -- no single required framework.
- Each agent has its own **specialized tools and instructions**.

---

## Next Lab

**Lab 04-01: PyFunc Chain** (Module 04 -- Assembling and Deploying) -- you
will learn how to package a complete RAG pipeline as an `mlflow.pyfunc` model,
deploy it to a serving endpoint, and call it via REST API. This is where
your chains go from notebooks to production.